# Co-training on Google Colab (GPU)

**Before you start (on your Mac):** from `promptlab-v2` run `./scripts/package_for_colab.sh` — it writes `promptlab-v2-colab.zip` next to the project folder (no `.venv`, no `data/cache/`, no `outputs/`).

**Runtime:** use a **GPU** runtime (T4 or better). **Python 3.11+** is ideal (`pyproject.toml` asks for `>=3.11`). If your runtime is 3.10, try switching runtime type or install may still work.

**Secrets:** you need **`GROQ_API_KEY`** (Groq) and **`HF_TOKEN`** (Hugging Face — gated Pangram EditLens checkpoint + dataset license accepted on the hub).

## 1) Optional: Google Drive (large zip or persistent outputs)
Skip if you upload the zip through the Colab **Files** panel (≈100MB+).

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# ZIP = "/content/drive/MyDrive/promptlab-v2-colab.zip"  # then unzip in cell 3

## 2) Upload `promptlab-v2-colab.zip`
Run the next cell and pick the zip from your machine **or** place it on Drive and `!unzip` from there.

In [ ]:
from google.colab import files
import os

UPLOADED = list(files.upload().keys())
assert UPLOADED, "No file uploaded — select promptlab-v2-colab.zip"
ZIP_NAME = UPLOADED[0]
print("Uploaded:", ZIP_NAME)

In [ ]:
import shutil
import zipfile

ROOT = "/content/promptlab-v2"
shutil.rmtree(ROOT, ignore_errors=True)

with zipfile.ZipFile(ZIP_NAME, "r") as z:
    z.extractall("/content")

assert os.path.isdir(ROOT), f"Expected {ROOT} after unzip — check zip top folder name"
print("Ready:", ROOT)

## 3) Install dependencies
Colab already has PyTorch with CUDA; we install the rest and the project in editable mode.

In [ ]:
%%bash
set -e
cd /content/promptlab-v2
pip install -q hatchling
pip install -q -e .
python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

## 4) API tokens
**Option A — Colab secrets (recommended):** Settings → Secrets → add `HF_TOKEN` and `GROQ_API_KEY`.

**Option B — prompt:** uncomment the `getpass` lines and paste once per session.

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except Exception as e:
    print("Secrets not set via userdata:", e)

# from getpass import getpass
# os.environ["HF_TOKEN"] = getpass("HF_TOKEN: ")
# os.environ["GROQ_API_KEY"] = getpass("GROQ_API_KEY: ")

assert os.environ.get("GROQ_API_KEY"), "Set GROQ_API_KEY (secret or getpass)"
assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN for pangram/editlens_roberta-large"
print("Tokens present (values hidden).")

## 5) Sanity check: data + GPU
`cotrain` needs **`data/merged/train.jsonl`** and **`data/merged/val.jsonl`** (paths from `configs/detector.yaml`). They are included in the Colab zip if present on your machine when you zipped.

In [ ]:
from pathlib import Path

REPO = Path("/content/promptlab-v2")
train_p = REPO / "data" / "merged" / "train.jsonl"
val_p = REPO / "data" / "merged" / "val.jsonl"
print("train exists:", train_p.is_file(), train_p)
print("val exists:", val_p.is_file(), val_p)

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 6) Smoke run (tiny budget)
One round, small population — checks Groq + detector download + one retrain. Remove `--smoke` for a real run (long + many API calls).

In [ ]:
%%bash
set -e
cd /content/promptlab-v2
export PYTHONPATH=/content/promptlab-v2
export PYTHONUNBUFFERED=1
python cotrain/loop.py --smoke

## 7) Full co-training (optional)
Edit `configs/cotrain.yaml` first if you want fewer topics / smaller models for rate limits. Then:

In [ ]:
%%bash
set -e
cd /content/promptlab-v2
export PYTHONPATH=/content/promptlab-v2
export PYTHONUNBUFFERED=1
# python cotrain/loop.py --config configs/cotrain.yaml

## 8) Download artifacts for your Mac / report
Zips `outputs/` (detector rounds, `metrics_final.json`, cotrain summaries) for `cotrain_report.ipynb`.

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

REPO = Path("/content/promptlab-v2")
out = REPO / "outputs"
zip_path = "/content/cotrain_outputs.zip"
if out.is_dir():
    shutil.make_archive("/content/cotrain_outputs", "zip", str(REPO), "outputs")
    files.download(zip_path)
else:
    print("No outputs/ yet — run co-training first.")